###PreClass Setup

In [7]:
#Run this at the start of every notebook this week
from google.colab import userdata
import subprocess
try:
  from groq import Groq
except ImportError:
  subprocess.run(["pip","install","groq","-q"])
  from groq import Groq

for pkg in ["chromadb","sentence-transformers","pypdf"]:
  subprocess.run(["pip","install",pkg, "-q"])
print("RAG libraries ready!")

GROQ_API_KEY = userdata.get("kaustubh_aiml")

client = Groq(api_key=GROQ_API_KEY)
model = "openai/gpt-oss-120b"

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role":"user","content":"Say Hello"}
    ]
)

RAG libraries ready!


###Day 1

####Why RAG Exists - The Two Problems Every LLM Has

In [ ]:
#Proving the knowledge gap Live
def ask(question):
  response = client.chat.completions.create(
      model=model,
      messages = [{"role":"user","content":question}]
  )
  return response.choices[0].message.content

#QUESTION 1: Something recent, beyond training data
print(ask("What were the exact results of yesterday's football match"
          "between two teams I haven't specified"))

print("\n" + "=" * 55 + "\n")

#QUESTION 1: Something private, about YOUR company
print(ask("What is TechCorp's internal remote work policy regarding"
            "how many days per month employees can work from home?"))

I’m happy to look up the match details for you, but I need to know which teams (or which league/competition) you’re interested in. Could you let me know the two teams or the specific match you’d like the result for?


I’m sorry, but I can’t help with that.


###RAG PIPELINE

####Text Embeddings For Documents

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading a sentence embedding model (similar spirit to week 6's)")
print("Word embeddings, but for WHOLE SENTENCES, not single words)...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "Employees can work from home up to 3 days per week.",
    "Remote work policy allows 3 days of home office weekly.",
    "The company cafeteria serves lunch from 12pm to 2pm.",
    "Our vacation policy grants 20 days of paid leave annually.",
]
embeddings = embedder.encode(sentences)
print(f"\nEach sentence became a vector of {embeddings.shape[1]} numbers")
print(f"Embeddings Shape: {embeddings.shape}")

print(f"\nFirst 10 numbers of sentences 1's embedding")
print(embeddings[0][:10])

print(f"""
NOTICE THE CONNECTION TO WEEK 6:
Just like work embeddings placed SIMILAR WORDS close together in
space, sentences embeddings place SIMILAR MEANING SENTENCES close
together -- even if they use completely DIFFERENT WORDS.

Sentence 1 and Sentence 2 above say almost the SAME thing USING
DIFFERENT WORDS -- let's verify their embeddings are close
""")

Loading a sentence embedding model (similar spirit to week 6's)
Word embeddings, but for WHOLE SENTENCES, not single words)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Each sentence became a vector of 384 numbers
Embeddings Shape: (4, 384)

First 10 numbers of sentences 1's embedding
[ 0.0049118  -0.03571659  0.00460999 -0.01953214 -0.03049482  0.00831898
 -0.04284544 -0.07611324 -0.12584074 -0.05580918]

NOTICE THE CONNECTION TO WEEK 6:
Just like work embeddings placed SIMILAR WORDS close together in
space, sentences embeddings place SIMILAR MEANING SENTENCES close
together -- even if they use completely DIFFERENT WORDS.

Sentence 1 and Sentence 2 above say almost the SAME thing USING
DIFFERENT WORDS -- let's verify their embeddings are close



####Semantic Search From Scratch

In [ ]:
def cosine_similarity(vec1,vec2):
  """Measure how SIMILAR two vector's directions are (not their length)"""
  dot_product = np.dot(vec1,vec2)
  magnitude   = np.linalg.norm(vec1) * np.linalg.norm(vec2)
  return dot_product / magnitude

print("COSINE SIMILARITY -- MEASURING MEANING-CLOSENESS")
print("=" * 55)
print("""
ANALOGY: Imagine two arrows pointing from the center of a room.
If they point in almost the SAME DIRECTION, they're similar --
regardless of how LONG each arrow is. Cosine similarity measures
the ANGLE between two vectors, ignoring their magnitude/length.

Score interpretation:
  1.0 = pointing in EXACTLY the same direction (identical meaning)
  0.0 = pointing in completely unrelated directions
 -1.0 = pointing in OPPOSITE directions (rare for text embeddings)
""")

print("SIMILARITY MATRIX:")
print(f"{'':>50}", end="")
for i in range(len(sentences)):
    print(f"S{i+1:<8}", end="")
print()

for i, sent1 in enumerate(sentences):
    print(f"S{i+1}: {sent1[:45]:<45} ", end="")
    for j, sent2 in enumerate(sentences):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"{sim:<9.4f}", end="")
    print()



COSINE SIMILARITY -- MEASURING MEANING-CLOSENESS

ANALOGY: Imagine two arrows pointing from the center of a room.
If they point in almost the SAME DIRECTION, they're similar --
regardless of how LONG each arrow is. Cosine similarity measures
the ANGLE between two vectors, ignoring their magnitude/length.

Score interpretation:
  1.0 = pointing in EXACTLY the same direction (identical meaning)
  0.0 = pointing in completely unrelated directions
 -1.0 = pointing in OPPOSITE directions (rare for text embeddings)

SIMILARITY MATRIX:
                                                  S1       S2       S3       S4       
S1: Employees can work from home up to 3 days per 1.0000   0.7527   0.2539   0.3425   
S2: Remote work policy allows 3 days of home offi 0.7527   1.0000   0.1535   0.4611   
S3: The company cafeteria serves lunch from 12pm  0.2539   0.1535   1.0000   0.1181   
S4: Our vacation policy grants 20 days of paid le 0.3425   0.4611   0.1181   1.0000   


####Build a Mini Search Engine

In [ ]:
def semantic_search(query, documents, doc_embeddings, embedder, top_k=2):
    """A complete, minimal semantic search function"""
    query_embedding = embedder.encode([query])[0]

    similarities = [
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]

    top_indices = np.argsort(similarities)[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "document": documents[idx],
            "similarity": similarities[idx]
        })

    return results

test_queries = [
    "How many days can I work remotely?",
    "What time can I get food at work?",
    "How much time off do I get per year?",
]

for query in test_queries:
    print(f"Query: '{query}'")
    results = semantic_search(query, sentences, embeddings, embedder, top_k=1)

    for r in results:
        print(f"  Best match ({r['similarity']:.3f}): {r['document']}")
    print()

print("NOTICE: none of these queries use the EXACT wording from the")
print("source sentences, yet semantic search correctly finds the")
print("relevant document. This is the RETRIEVAL step of RAG,")
print("built completely from scratch!")

Query: 'How many days can I work remotely?'
  Best match (0.747): Remote work policy allows 3 days of home office weekly.

Query: 'What time can I get food at work?'
  Best match (0.559): The company cafeteria serves lunch from 12pm to 2pm.

Query: 'How much time off do I get per year?'
  Best match (0.398): Our vacation policy grants 20 days of paid leave annually.

NOTICE: none of these queries use the EXACT wording from the
source sentences, yet semantic search correctly finds the
relevant document. This is the RETRIEVAL step of RAG,
built completely from scratch!


###Day 2

####Building the Complete RAG PipeLine - Chunking + Vector Databases

####Document Chunking

In [4]:
# Document Chunking
company_document = """
TechCorp Employee Handbook - 2026 Edition

SECTION 1: REMOTE WORK POLICY
Employees are permitted to work remotely up to 3 days per week,
subject to manager approval. Remote work requests must be submitted
through the HR portal at least 48 hours in advance. Employees in
customer-facing roles may have additional restrictions based on
business needs.

SECTION 2: VACATION AND PAID TIME OFF
All full-time employees receive 20 days of paid vacation annually,
accrued at a rate of 1.67 days per month. Vacation days do not
expire and can be carried over to the following year, up to a
maximum of 10 unused days.

SECTION 3: HEALTH AND WELLNESS BENEFITS
TechCorp provides comprehensive health insurance covering medical,
dental, and vision care, with the company covering 80% of premium
costs. Employees also have access to a wellness stipend of $50 per
month for gym memberships or fitness classes.

SECTION 4: EXPENSE REIMBURSEMENT
Business-related expenses must be submitted within 30 days of the
expense date through the expense management system. Receipts are
required for all expenses over $25.
"""

def chunk_text(text, chunk_size = 400, overlap = 50):
  """Split text into overlapping chunks"""
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk.strip())
    start += (chunk_size - overlap)
  return [c for c in chunks if len(c) > 20]

chunks = chunk_text(company_document, chunk_size = 400, overlap = 50)

print(f"Original document: {len(company_document)} characters")
print(f"Split into {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
  print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
  print(chunk[:150] + "...")
  print()

Original document: 1097 characters
Split into 4 chunks

--- Chunk 1 (399 chars) ---
TechCorp Employee Handbook - 2026 Edition

SECTION 1: REMOTE WORK POLICY
Employees are permitted to work remotely up to 3 days per week,
subject to ma...

--- Chunk 2 (398 chars) ---
SECTION 2: VACATION AND PAID TIME OFF
All full-time employees receive 20 days of paid vacation annually,
accrued at a rate of 1.67 days per month. Vac...

--- Chunk 3 (396 chars) ---
overing medical,
dental, and vision care, with the company covering 80% of premium
costs. Employees also have access to a wellness stipend of $50 per
...

--- Chunk 4 (46 chars) ---
ceipts are
required for all expenses over $25....



####ChromaDB setup

In [8]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma_client = chromadb.Client()
collection    = chroma_client.create_collection(name="company_handbook")

embedder      = SentenceTransformer("all-MiniLM-L6-v2")

chunk_ids     = [f"chunk_{i}" for i in range(len(chunks))]
embeddings_list = embedder.encode(chunks).tolist()

collection.add(
    ids = chunk_ids,
    embeddings=embeddings_list,
    documents=chunks
)

print(f"Added {len(chunks)} chunks to ChromaDB collection 'company_handbook")
print(f"Total items in collection: {collection.count()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Added 4 chunks to ChromaDB collection 'company_handbook
Total items in collection: 4


####Query the Vector Database

In [12]:
def search_documents(query, n_results=2):
  """Query ChromaDB for the most relevant chunks"""
  query_embedding = embedder.encode([query]).tolist()
  result = collection.query(query_embeddings=query_embedding, n_results=n_results)
  return result

test_queries = [
    "How many vacation days do I get?",
    "Can I work from home?",
    "What is covered by health insurance",
    "How do I submit an expense report",
]

for query in test_queries:
  print(f"Query:  '{query}' ")
  results = search_documents(query,n_results=1)
  top_chunk = results["documents"][0][0]
  distance  = results["distances"][0][0]
  print(f"  Retrieved chunk(distance={distance:.3f}):")
  print(f"  {top_chunk[:200]}...")
  print()

print("NOTICE: ChromaDB returns a 'distance' score -- LOWER distance")
print("means MORE similar (opposite direction from cosine similarity,")
print("but measuring the same underlying concept).")

Query:  'How many vacation days do I get?' 
  Retrieved chunk(distance=0.854):
  SECTION 2: VACATION AND PAID TIME OFF
All full-time employees receive 20 days of paid vacation annually,
accrued at a rate of 1.67 days per month. Vacation days do not
expire and can be carried over t...

Query:  'Can I work from home?' 
  Retrieved chunk(distance=1.272):
  TechCorp Employee Handbook - 2026 Edition

SECTION 1: REMOTE WORK POLICY
Employees are permitted to work remotely up to 3 days per week,
subject to manager approval. Remote work requests must be submi...

Query:  'What is covered by health insurance' 
  Retrieved chunk(distance=1.303):
  overing medical,
dental, and vision care, with the company covering 80% of premium
costs. Employees also have access to a wellness stipend of $50 per
month for gym memberships or fitness classes.

SEC...

Query:  'How do I submit an expense report' 
  Retrieved chunk(distance=1.407):
  overing medical,
dental, and vision care, with the company covering 

####Complete RAG: Retrieve +Generate

In [18]:
def rag_query(question, n_chunks=2):
  """The COMPLETE RAG Pipeline: retrieve relevant chunks, then generate an answer"""

  results = search_documents(question, n_results=n_chunks)
  retrieved_chunks = results["documents"][0]
  context = "\n\n".join(retrieved_chunks)

  prompt = f"""Answer the question using ONLY the context provided below.
If the answer cannot be found in the context, say " I don't have that
information in the provided documents"

Context:
{context}

Question: {question}

Answer:"""

  response = client.chat.completions.create(
      model = model,
      messages = [{"role":"user","content":prompt}],
      temperature=0
      )
  return response.choices[0].message.content, retrieved_chunks

print("COMPLETE RAG SYSTEM --  Company Handbook Q&A")
print("="*55)

questions = [
    "How many vacation days do I get, can can I carry them over?",
    "What's the process for working remotely",
    "Does the company offer mental health support",
    "What is the company's policy on cryptocurrency investments",
]

for question in questions:
  print(f"\nQ:  {question}")
  answer, sources = rag_query(question)
  print(f"A:  {answer}")

COMPLETE RAG SYSTEM --  Company Handbook Q&A

Q:  How many vacation days do I get, can can I carry them over?
A:  You receive **20 paid vacation days per year** (accruing at about **1.67 days each month**). Vacation days do not expire, and you may **carry over up to 10 unused days** into the next year.

Q:  What's the process for working remotely
A:  To work remotely, you must:

1. **Submit a request** through the HR portal **at least 48 hours in advance**.  
2. **Obtain manager approval** for the request.  
3. You may work remotely **up to 3 days per week**, unless you’re in a customer‑facing role, which may have additional restrictions based on business needs.

Q:  Does the company offer mental health support
A:  I don't have that information in the provided documents.

Q:  What is the company's policy on cryptocurrency investments
A:  I don't have that information in the provided documents.
